# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

In [ ]:
<h1> Tru A. Proposal (ECO590)</h1>

<h2> Proposal/Research Question: Do sustainability signals in fashion act as a pricing strategy? (ASOS)</h2>

<h4>Data Source</h4>
This project will use webscraped data from ASOS. This is a global online fashion retail store, headquartered in London.
Two datasets will be collected from publicly available product listing pages:
- Women’s tops category (non-sustainable products)
- Search results for ASOS’s “Circular Design Collection” (sustainable products)

<h4>Links:</h4>

[ASOS Women's Tops](https://www.asos.com/women/tops/cat/?cid=4169)

[ASOS Circular Design Collection](https://www.asos.com/us/search/?q=circular+design+collection&currentpricerange=10-170&refine=floor%3A1001%2C2001)

<h4>More about the circular design collection:(basically just sustainable clothing line ASOS made)</h4>

[Circulare Design Info](https://www.asos.com/discover/circular-design/?msockid=3ba92ad05ebf620415693e225f0f6303)

No API key is required, as the data is fully accessible through ASOS’s public website.

<h4>Variables</h4>

1. Price
This is a continuous variable
Will be extracted from product listing pages
2. Category (A Constructed Variable)
From the product names (e.g., tops, jeans, dresses, shorts)
This is used to control for differences across product types
3. Sustainable (Marketing Signal Variable)
Binary variable (1 = Circular Design Collection, 0 = not included)
Was made based on whether a product belongs to ASOS’s Circular Design Collection

This variable captures whether a product is marketed as sustainable, rather than measuring sustainability directly. It is not provided explicitly in the dataset and is constructed using scraped information

<h4>Motivation</h4>
I'm interested in researching fashion and the environment and I'd also like to dive into marketing. My past research as been 
done on consumer behavior, fashion items, and the environment/sustainability. I think marketing is a good thing to focus on so
I can help brands learn how to market sustainable items and how certain sustainability marketing tactics affect consumer behavior.
This project will help see if sustainable products are typically more expensive, which can be used for marketing and how to convince
consumers whatever price premium there is, is worth it. From a marketing perspective, sustainability may function as a signal of
product quality, ethical production, or brand value, potentially allowing firms to position these items at higher price points.
This project examines whether products marketed as sustainable (ASOS’s Circular Design Collection) are priced differently from
non-sustainable products.

In [ ]:
!pip install bs4 requests pandas

In [ ]:
import requests #sends http requests to get data from the site
import pandas as pd #stores cleans and merges data
import time #used this to slow thigns down so i wouldn;t get blocked
from bs4 import BeautifulSoup #extracts information
import re #gets numbers from texts
import random #randomizes things to mimic humans
#importing the libraries I need

In [ ]:
#Womens Tops

#User Agent to prevent blocking/Telling site who I am
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
]

#Women's Top url/first page
base_url = "https://www.asos.com/women/tops/cat/?cid=4169&page="

#empty list for products
products = []
session = requests.Session()

for page in range(1, 6):  # 5 pages ≈ abt 72 items per page 
    url = base_url + str(page) #each new page the page number increases
    
    success = False #tracks if the scraping worked or not. Turns to True if it works
    
    for attempt in range(3): #retry loop. will try 3 times with the same page if it doesn't work
        try:         #was running into some problems earlier so used this to have a more likely chance of success
            headers = {
                "User-Agent": random.choice(user_agents)
            }  #this also helps avoid being blocked, randomly picks a different browser identity each time
            
            response = session.get(url, headers=headers, timeout=20) #makes site wait 20 seconds before failing it
            soup = BeautifulSoup(response.text, "html.parser") #changes the html to something I can webscrape
            
            items = soup.find_all("li", class_=lambda x: x and "productTile" in x) #finds all the matching product blocks
            #li is one item. lambda is also kind of telling it to find what contains the tag
            
            if len(items) == 0:
                raise Exception("No items found") #if nothing is found, makes it retry
            
            for item in items: #looping through the products
                
                #PRODUCT NAME
                name_tag = item.find("p", class_=lambda x: x and "productDescription" in x) #find the name
                name = name_tag.text.strip() if name_tag else None #take out extra spaces and if it isn't found itll say none
                
                # PRICE
                price_tag = item.find("p", attrs={"aria-label": True}) #finds the price
                
                if price_tag: #if the price is there...
                    raw_price = price_tag.get("aria-label") #should give the number price 
                    match = re.search(r"\d+\.?\d*", raw_price) #regx takes out the number no symbol attached
                    price = float(match.group()) if match else None #turns number into a float
                else:
                    price = None #again if there's no price it'll say none
                
                # LINK
                link_tag = item.find("a", href=True) #finding the link
                link = link_tag["href"] if link_tag else None #takes out URL
                
                products.append({
                    "name": name,
                    "price": price,
                    "link": link
                })#adding info to the empty list
            
            print(f"Finished page {page}") #tells me if the page was scraped successfully
            success = True
            break #doesn't try the loop again if success is True
        
        except Exception as e: #if it were to fail...
            print(f"Retry {attempt+1} failed for page {page}: {e}") #it would print the error message
            time.sleep(random.uniform(5, 10)) #waits 5-10 secs before redo. also helps not getting blocked
    
    if not success:
        print(f"Skipping page {page}") #will skip the page if all of the 3 redos dont work
    
    time.sleep(random.uniform(5, 8)) #wait 5-8 secs before scraping each page. makes it go slower so not as suspicious

# CREATE DATAFRAME
df = pd.DataFrame(products) #makes data table
df_tops = df.copy()
df_tops["sustainable"] = 0

print(df.head()) #shows us first 5 rows
print("Total items:", len(df)) #shows number amount total

In [ ]:
df_regular = df.copy()
df_regular["sustainable"] = 0

In [ ]:
#CIRCULAR DESIGN COLLECTION CODE, same code as above. Only change is the base url and number of pages ran

# USER AGENTS (prevents blocking)
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
]

base_url = "https://www.asos.com/us/search/?q=circular+design+collection&page="

products = []
session = requests.Session()

for page in range(1, 3):  # 2 pages ≈ 100+ items (collection doesn't have a bunch of items)
    url = base_url + str(page)
    
    success = False
    
    for attempt in range(3):
        try:
            headers = {
                "User-Agent": random.choice(user_agents)
            }
            
            response = session.get(url, headers=headers, timeout=20)
            soup = BeautifulSoup(response.text, "html.parser")
            
            items = soup.find_all("li", class_=lambda x: x and "productTile" in x)
            
            if len(items) == 0:
                raise Exception("No items found")
            
            for item in items:
                
                # PRODUCT NAME
                name_tag = item.find("p", class_=lambda x: x and "productDescription" in x)
                name = name_tag.text.strip() if name_tag else None
                
                # PRICE
                price_tag = item.find("p", attrs={"aria-label": True})
                
                if price_tag:
                    raw_price = price_tag.get("aria-label")
                    match = re.search(r"\d+\.?\d*", raw_price)
                    price = float(match.group()) if match else None
                else:
                    price = None
                
                # LINK
                link_tag = item.find("a", href=True)
                link = link_tag["href"] if link_tag else None
                
                products.append({
                    "name": name,
                    "price": price,
                    "link": link
                })
            
            print(f"Finished page {page}")
            success = True
            break
        
        except Exception as e:
            print(f"Retry {attempt+1} failed for page {page}: {e}")
            time.sleep(random.uniform(5, 10))
    
    if not success:
        print(f"Skipping page {page}")
    
    time.sleep(random.uniform(5, 8))

# CREATE DATAFRAME
df = pd.DataFrame(products)
df_circular = df.copy()
df_circular["sustainable"] = 1

print(df.head())
print("Total items:", len(df))

In [ ]:
#Womens dress code, same code different url

#User Agent to prevent blocking/Telling site who I am
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
]

#Women's Dresses url/first page
base_url = "https://www.asos.com/us/women/dresses/cat/?cid=8799&page="

#empty list for products
products = []
session = requests.Session()

for page in range(1, 6):  # 5 pages ≈ abt 72 items per page 
    url = base_url + str(page) #each new page the page number increases
    
    success = False #tracks if the scraping worked or not. Turns to True if it works
    
    for attempt in range(3): #retry loop. will try 3 times with the same page if it doesn't work
        try:         #was running into some problems earlier so used this to have a more likely chance of success
            headers = {
                "User-Agent": random.choice(user_agents)
            }  #this also helps avoid being blocked, randomly picks a different browser identity each time
            
            response = session.get(url, headers=headers, timeout=20) #makes site wait 20 seconds before failing it
            soup = BeautifulSoup(response.text, "html.parser") #changes the html to something I can webscrape
            
            items = soup.find_all("li", class_=lambda x: x and "productTile" in x) #finds all the matching product blocks
            #li is one item. lambda is also kind of telling it to find what contains the tag
            
            if len(items) == 0:
                raise Exception("No items found") #if nothing is found, makes it retry
            
            for item in items: #looping through the products
                
                #PRODUCT NAME
                name_tag = item.find("p", class_=lambda x: x and "productDescription" in x) #find the name
                name = name_tag.text.strip() if name_tag else None #take out extra spaces and if it isn't found itll say none
                
                # PRICE
                price_tag = item.find("p", attrs={"aria-label": True}) #finds the price
                
                if price_tag: #if the price is there...
                    raw_price = price_tag.get("aria-label") #should give the number price 
                    match = re.search(r"\d+\.?\d*", raw_price) #regx takes out the number no symbol attached
                    price = float(match.group()) if match else None #turns number into a float
                else:
                    price = None #again if there's no price it'll say none
                
                # LINK
                link_tag = item.find("a", href=True) #finding the link
                link = link_tag["href"] if link_tag else None #takes out URL
                
                products.append({
                    "name": name,
                    "price": price,
                    "link": link
                })#adding info to the empty list
            
            print(f"Finished page {page}") #tells me if the page was scraped successfully
            success = True
            break #doesn't try the loop again if success is True
        
        except Exception as e: #if it were to fail...
            print(f"Retry {attempt+1} failed for page {page}: {e}") #it would print the error message
            time.sleep(random.uniform(5, 10)) #waits 5-10 secs before redo. also helps not getting blocked
    
    if not success:
        print(f"Skipping page {page}") #will skip the page if all of the 3 redos dont work
    
    time.sleep(random.uniform(5, 8)) #wait 5-8 secs before scraping each page. makes it go slower so not as suspicious

# CREATE DATAFRAME
df = pd.DataFrame(products) #makes data table
df_dresses = df.copy()
df_dresses["sustainable"] = 0

print(df.head()) #shows us first 5 rows
print("Total items:", len(df)) #shows number amount total

In [ ]:
#Womens shorts, same code different url

#User Agent to prevent blocking/Telling site who I am
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
]

#Women's Shorts url/first page
base_url = "https://www.asos.com/us/search/?q=shorts&currentpricerange=0-775&refine=floor:1000"

#empty list for products
products = []
session = requests.Session()

for page in range(1, 6):  # 5 pages ≈ abt 72 items per page 
    url = base_url + str(page) #each new page the page number increases
    
    success = False #tracks if the scraping worked or not. Turns to True if it works
    
    for attempt in range(3): #retry loop. will try 3 times with the same page if it doesn't work
        try:         #was running into some problems earlier so used this to have a more likely chance of success
            headers = {
                "User-Agent": random.choice(user_agents)
            }  #this also helps avoid being blocked, randomly picks a different browser identity each time
            
            response = session.get(url, headers=headers, timeout=20) #makes site wait 20 seconds before failing it
            soup = BeautifulSoup(response.text, "html.parser") #changes the html to something I can webscrape
            
            items = soup.find_all("li", class_=lambda x: x and "productTile" in x) #finds all the matching product blocks
            #li is one item. lambda is also kind of telling it to find what contains the tag
            
            if len(items) == 0:
                raise Exception("No items found") #if nothing is found, makes it retry
            
            for item in items: #looping through the products
                
                #PRODUCT NAME
                name_tag = item.find("p", class_=lambda x: x and "productDescription" in x) #find the name
                name = name_tag.text.strip() if name_tag else None #take out extra spaces and if it isn't found itll say none
                
                # PRICE
                price_tag = item.find("p", attrs={"aria-label": True}) #finds the price
                
                if price_tag: #if the price is there...
                    raw_price = price_tag.get("aria-label") #should give the number price 
                    match = re.search(r"\d+\.?\d*", raw_price) #regx takes out the number no symbol attached
                    price = float(match.group()) if match else None #turns number into a float
                else:
                    price = None #again if there's no price it'll say none
                
                # LINK
                link_tag = item.find("a", href=True) #finding the link
                link = link_tag["href"] if link_tag else None #takes out URL
                
                products.append({
                    "name": name,
                    "price": price,
                    "link": link
                })#adding info to the empty list
            
            print(f"Finished page {page}") #tells me if the page was scraped successfully
            success = True
            break #doesn't try the loop again if success is True
        
        except Exception as e: #if it were to fail...
            print(f"Retry {attempt+1} failed for page {page}: {e}") #it would print the error message
            time.sleep(random.uniform(5, 10)) #waits 5-10 secs before redo. also helps not getting blocked
    
    if not success:
        print(f"Skipping page {page}") #will skip the page if all of the 3 redos dont work
    
    time.sleep(random.uniform(5, 8)) #wait 5-8 secs before scraping each page. makes it go slower so not as suspicious

# CREATE DATAFRAME
df = pd.DataFrame(products) #makes data table
df_shorts = df.copy()
df_shorts["sustainable"] = 0

print(df.head()) #shows us first 5 rows
print("Total items:", len(df)) #shows number amount total

In [ ]:
#Womens Jeans Same code with jean base url

#User Agent to prevent blocking/Telling site who I am
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
]

#Women's Jeans url/first page
base_url = "https://www.asos.com/us/women/jeans/cat/?cid=3630&page="

#empty list for products
products = []
session = requests.Session()

for page in range(1, 6):  # 5 pages ≈ abt 72 items per page 
    url = base_url + str(page) #each new page the page number increases
    
    success = False #tracks if the scraping worked or not. Turns to True if it works
    
    for attempt in range(3): #retry loop. will try 3 times with the same page if it doesn't work
        try:         #was running into some problems earlier so used this to have a more likely chance of success
            headers = {
                "User-Agent": random.choice(user_agents)
            }  #this also helps avoid being blocked, randomly picks a different browser identity each time
            
            response = session.get(url, headers=headers, timeout=20) #makes site wait 20 seconds before failing it
            soup = BeautifulSoup(response.text, "html.parser") #changes the html to something I can webscrape
            
            items = soup.find_all("li", class_=lambda x: x and "productTile" in x) #finds all the matching product blocks
            #li is one item. lambda is also kind of telling it to find what contains the tag
            
            if len(items) == 0:
                raise Exception("No items found") #if nothing is found, makes it retry
            
            for item in items: #looping through the products
                
                #PRODUCT NAME
                name_tag = item.find("p", class_=lambda x: x and "productDescription" in x) #find the name
                name = name_tag.text.strip() if name_tag else None #take out extra spaces and if it isn't found itll say none
                
                # PRICE
                price_tag = item.find("p", attrs={"aria-label": True}) #finds the price
                
                if price_tag: #if the price is there...
                    raw_price = price_tag.get("aria-label") #should give the number price 
                    match = re.search(r"\d+\.?\d*", raw_price) #regx takes out the number no symbol attached
                    price = float(match.group()) if match else None #turns number into a float
                else:
                    price = None #again if there's no price it'll say none
                
                # LINK
                link_tag = item.find("a", href=True) #finding the link
                link = link_tag["href"] if link_tag else None #takes out URL
                
                products.append({
                    "name": name,
                    "price": price,
                    "link": link
                })#adding info to the empty list
            
            print(f"Finished page {page}") #tells me if the page was scraped successfully
            success = True
            break #doesn't try the loop again if success is True
        
        except Exception as e: #if it were to fail...
            print(f"Retry {attempt+1} failed for page {page}: {e}") #it would print the error message
            time.sleep(random.uniform(5, 10)) #waits 5-10 secs before redo. also helps not getting blocked
    
    if not success:
        print(f"Skipping page {page}") #will skip the page if all of the 3 redos dont work
    
    time.sleep(random.uniform(5, 8)) #wait 5-8 secs before scraping each page. makes it go slower so not as suspicious

# CREATE DATAFRAME
df = pd.DataFrame(products) #makes data table
df_jeans = df.copy()
df_jeans["sustainable"] = 0

print(df.head()) #shows us first 5 rows
print("Total items:", len(df)) #shows number amount total

In [ ]:
df_final = pd.concat([df_tops, df_dresses, df_jeans, df_shorts, df_circular], ignore_index=True) #merging datasets

def get_category(name): #assigns category based on name
    if not name:
        return "other" #it'll say other if not apart of the list
    
    name = name.lower() #makes all lowercase
    
    if "top" in name: #if one of these words are in the title of the item it'll categorize it as that
        return "top" #I have to watch out for "Topshop" though because the computer recognizes that as tops and not a brand
    elif "jeans" in name:
        return "jeans"
    elif "dress" in name:
        return "dress"
    elif "shorts" in name:
        return "shorts"
    else:
        return "other"

df_final["category"] = df_final["name"].apply(get_category) #stores name in category column

df_final = df_final.drop_duplicates(subset="name") #takes out duplicates to decrease errors

In [ ]:
df_final.to_csv("asos_sustainability_data.csv", index=False) #export to csv

In [ ]:
Source: https://carolinamendozab.medium.com/scrapping-price-data-from-a-website-using-beautiful-soup-e91fb6e8cb88
https://thunderbit.com/blog/web-scraping-without-getting-blocked

I also would just look things up, answers would pop up in the text/google box